# SENA V13 - DWN matched baseline on Colab (fallback)

Runtime -> Change runtime type -> **T4 GPU**. Zip the package folder (`Baseline_DWN_pkg`) as `Baseline_DWN.zip` and put it in your Google Drive root first.

Nothing in this notebook is a result until you run it. Record the outputs listed in README.md.

In [ ]:
!nvidia-smi
import torch, sys; print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/Baseline_DWN.zip /content/ && cd /content && unzip -q -o Baseline_DWN.zip -d /content/pkg
import glob, os
PKG = os.path.dirname(sorted(glob.glob('/content/pkg/**/train_dwn.py', recursive=True))[0])
%cd $PKG
!ls

## 1. Install the official DWN package (compiles its CUDA extension, ~3-6 min)
Pinned to the commit inspected for this package. Uses Colab's preinstalled PyTorch (`--no-build-isolation`).

In [ ]:
!pip install -q ninja
!pip install --no-build-isolation -v git+https://github.com/alanbacellar/DWN@2b456acfc7f679a8c8f6ea18835f47803aa5b0da 2>&1 | tail -n 5
!python -c "import torch_dwn, efd_cuda; print('torch_dwn + efd_cuda OK')"

In [ ]:
# Optional: check that the no-compile fallback matches the official CUDA kernel
!python train_dwn.py --check-efd

## 2. Data: verify SENA's exact split (MNIST, thr 0.3, 10k/2.5k, seed 2026)

In [ ]:
!python make_split.py --sena-npz data/mnist_bin_t0.3_tr10000_te2500_s2026.npz
!cat data/mnist_bin_t0.3_tr10000_te2500_s2026.sha256.json | head -n 20

## 3. Train the published small model (5 seeds, 100 epochs each)
If the extension did not build, replace `--efd ext` by `--efd torch`.

In [ ]:
!python train_dwn.py --arch sm --seeds 0 1 2 3 4 --efd ext 2>&1 | grep -E 'ep (10|20|30|40|50|60|70|80|90|100)/|label'

In [ ]:
# Optional larger variant (paper 'lg' layer sizes with 1-bit input)
# !python train_dwn.py --arch lg_z1 --seeds 0 1 2 3 4 --efd ext

## 4. Generate Verilog for one seed (seed 0) and check it bit-exactly with Icarus Verilog

In [ ]:
!apt-get -qq install -y iverilog > /dev/null
!python dwn_to_verilog.py runs/sm/seed0/model.json --out quartus/gen --sim

## 5. Accuracy / coverage tables, then save everything back to Drive
Quartus must be run on the PC (Windows) - copy `quartus/gen` there.

In [ ]:
!python collect_results.py --arch sm
!cat results/dwn_accuracy_summary.csv
!cat results/dwn_coverage.csv

In [ ]:
import time
tag = time.strftime('%Y%m%d_%H%M%S')
!zip -q -r /content/drive/MyDrive/DWN_results_{tag}.zip runs results quartus/gen
print('saved DWN_results_' + tag + '.zip to Drive')